# Chapter 9 &mdash; Checking the Conversion by Round-Tripping to Minimal DFA

**Concept 5 of the Chapter 9 decomposition:** *Checking the Conversion by Round-Tripping to Minimal DFA*

Convert the produced RE back to a minimal DFA and check `iso_dfa` against the original.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter9/Concept-Round-Trip-Check/Concept-Round-Trip-Check.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.Def_RE2NFA     import *
from jove.Def_NFA2RE     import *
from jove.AnimateDFA     import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


The conversion produces a large, unreadable RE. How do you know it is right?

**Round-trip it.** Convert the RE back with `re2nfa`, determinize, minimize, and
compare with the minimized original using `iso_dfa`. By Myhill&ndash;Nerode a `True`
answer means the languages are **identical** &mdash; not merely similar.

$$N \xrightarrow{\ \text{NFA2RE}\ } R \xrightarrow{\ \text{RE2NFA}\ } N' \xrightarrow{\ \text{det, min}\ } D' \;\cong\; \text{min}(N)$$

This is the standard way to test *any* language-preserving transformation, and it
costs four function calls.

## 2. Definitions

### The round trip, as one function

In [ ]:
def round_trip(N):
    _, _, r = del_gnfa_states(mk_gnfa(N))
    return r, min_dfa(nfa2dfa(N)), min_dfa(nfa2dfa(re2nfa(r)))

### A batch of machines to try it on

In [ ]:
MACHINES = {
 'even 0s'      : '''NFA
IF : 0 -> A
IF : 1 -> IF
A  : 0 -> IF
A  : 1 -> A
''',
 'ends in 01'   : '''NFA
I : 0 | 1 -> I
I : 0 -> A
A : 1 -> F
''',
 'contains 11'  : '''NFA
I : 0 | 1 -> I
I : 1 -> A
A : 1 -> F
F : 0 | 1 -> F
''',
 'third-last 1' : '''NFA
I : 0 | 1 -> I
I : 1 -> A
A : 0 | 1 -> B
B : 0 | 1 -> F
''',
}

## 3. Tests

Every machine round-trips to an isomorphic minimal DFA.

In [ ]:
results = {}
for name, src in MACHINES.items():
    N = md2mc(src)
    r, D0, D1 = round_trip(N)
    results[name] = (len(r), len(D0["Q"]), len(D1["Q"]), iso_dfa(D0, D1))
    print("%-15s RE len %4d  min |Q| %2d vs %2d  iso %s" % ((name,) + results[name]))
    assert iso_dfa(D0, D1)

String-level cross-check, for good measure.

In [ ]:
from itertools import product
strs = [''.join(p) for k in range(10) for p in product('01', repeat=k)]
for name, src in MACHINES.items():
    N = md2mc(src)
    r, _, D1 = round_trip(N)
    assert all(accepts_nfa(N, s) == accepts_dfa(D1, s) for s in strs)
    print("%-15s agrees on all %d strings" % (name, len(strs)))

A **broken** transformation is caught immediately.

In [ ]:
N = md2mc(MACHINES['ends in 01'])
r, D0, _ = round_trip(N)
broken = r + "0"                       # append a stray symbol to the RE
Db = min_dfa(nfa2dfa(re2nfa(broken)))
print("tampered RE round-trips to an isomorphic machine? ", iso_dfa(D0, Db))
assert not iso_dfa(D0, Db)
langeq_dfa(D0, Db, gen_counterex=True)

And the check is cheap: four calls, whatever the machine size.

In [ ]:
print("round_trip = del_gnfa_states + re2nfa + nfa2dfa + min_dfa")
print("cost is dominated by nfa2dfa, which is exponential in the worst case --")
print("but on the machines you actually write, it is instant.")

## 4. Animation

One of the round-tripped machines.

*(The `display(HTML(...))` line loads the toolbar's font-awesome icons. Keep it last in the cell &mdash; it must be there for the controls to appear.)*

In [ ]:
from jove.AnimateDFA import *
AnimateDFA(min_dfa(nfa2dfa(md2mc(MACHINES['contains 11']))), FuseEdges=True)
display(HTML('<link rel="stylesheet" href="//stackpath.bootstrapcdn.com/font-awesome/4.7.0/css/font-awesome.min.css"/>'))

## 5. Exercises


1. Round-trip a machine **twice**. Does the RE stabilise?
2. Why is `iso_dfa` the right comparison here rather than string testing?
3. What does the counterexample walk print for the tampered RE?

In [ ]:
# Your work for the exercises above.